In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

In [30]:
# Create a DataFrame from your provided data
data = {
    "Epoch": [50, 100, 200, 300, 400, 500],
    "ACC": [0.9354, 0.9790, 0.9677, 0.9054, 0.8869, 0.8585],
    "AUC": [0.9848, 0.9857, 0.9852, 0.9854, 0.9851, 0.9849],
    "PRE": [0.9017, 0.9066, 0.9044, 0.9019, 0.9031, 0.9052],
    "SP": [0.8979, 0.9029, 0.9005, 0.8980, 0.8997, 0.9011],
    "SN": [0.9686, 0.9040, 0.9693, 0.9686, 0.9691, 0.9696],
    "F1": [0.8993, 0.8795, 0.9020, 0.8995, 0.9009, 0.9025],
    "MCC": [0.8733, 0.9790, 0.8763, 0.8732, 0.8752, 0.8774]
}


In [31]:
df = pd.DataFrame(data)

In [32]:
# Reset matplotlib settings to default
plt.rcParams.update(plt.rcParamsDefault)

# Set Times New Roman font with fallback
plt.rcParams['font.family'] = ['Times New Roman', 'serif']
plt.rcParams['mathtext.fontset'] = 'stix'

# Set global plot style with large font sizes
plt.rcParams['font.size'] = 32
plt.rcParams['axes.labelsize'] = 36
plt.rcParams['axes.titlesize'] = 40
plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30
plt.rcParams['legend.fontsize'] = 22
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['savefig.dpi'] = 1000
plt.rcParams['figure.facecolor'] = 'white'

In [33]:
# Enhanced color palette for Epochs
epoch_palette = {
    50: '#FFBB78',   # Light Orange
    100: '#1F77B4',   # Bright Blue
    200: '#FF7F0E',   # Rich Orange
    300: '#2CA02C',   # Vivid Green
    400: '#D62728',   # Bold Red
    500: '#9467BD',   # Deep Purple
    1000: '#8C564B'   # Brown
}

# Metric palette
metric_palettes = {
    'ACC': 'viridis',
    'AUC': 'plasma',
    'PRE': 'cividis',
    'SP': 'magma',
    'SN': 'inferno',
    'F1': 'cividis',
    'MCC': 'plasma'
}

In [34]:
# Function to save figures in both PNG and PDF
def save_figure(fig, filename):
    output_folder = "Experiment_Output_Figures/Epoch"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    png_path = os.path.join(output_folder, f"{filename}.png")
    pdf_path = os.path.join(output_folder, f"{filename}.pdf")
    
    fig.savefig(png_path, dpi=1000, bbox_inches='tight', facecolor='white', format='png')
    fig.savefig(pdf_path, dpi=1000, bbox_inches='tight', facecolor='white', format='pdf')
    
    print(f"Saved: {png_path} and {pdf_path}")
    plt.close(fig)

In [35]:
# Function to generate the violin plot visualization
def generate_violin_plot():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    melted_df = pd.melt(df, id_vars=['Epoch'],
                       value_vars=metrics,
                       var_name='Metric',
                       value_name='Score')
    
    fig, ax = plt.subplots(figsize=(20, 14))
    sns.violinplot(x='Epoch', 
                   y='Score', 
                   hue='Epoch',
                   data=melted_df,
                   palette=epoch_palette, 
                   inner='box',
                   linewidth=2, 
                   ax=ax,
                   legend=False)
    
    ax.set_title('Distribution of Performance Scores by Epoch',
                fontsize=44, pad=30)
    ax.set_xlabel('Epoch', fontsize=40, labelpad=25)
    ax.set_ylabel('Score Distribution Across Metrics', fontsize=40, labelpad=25)
    ax.set_ylim(0.75, 1.1)  # Adjusted for your data range
    
    plt.tight_layout(pad=3.0)
    save_figure(fig, "epoch_violin_plot")

In [36]:
# Function to generate the timing comparison plot
def generate_timing_comparison():
    fig, ax = plt.subplots(figsize=(18, 10))
    sorted_df = df.sort_values(by='Training_Time', ascending=True)
    
    bars = ax.barh(sorted_df['Epoch'].astype(str), sorted_df['Training_Time'],
                  color=[epoch_palette[ep] for ep in sorted_df['Epoch']],
                  height=0.6)
    
    for i, epoch in enumerate(sorted_df['Epoch']):
        test_time = sorted_df[sorted_df['Epoch'] == epoch]['Testing_Time'].values[0]
        ax.text(5, i, f'Test: {test_time:.4f}s', ha='left', va='center',
                fontsize=22, color='black')
    
    ax.set_title('Training and Testing Time by Epoch', fontsize=40, pad=30)
    ax.set_xlabel('Training Time (seconds)', fontsize=36, labelpad=25)
    ax.set_ylabel('Epoch', fontsize=36, labelpad=25)
    
    max_time = sorted_df['Training_Time'].max()
    ax.set_xlim(0, max_time * 1.2)
    
    plt.tight_layout(pad=3.0)
    save_figure(fig, "epoch_timing_comparison")

In [37]:
metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
melted_df = pd.melt(df, id_vars=['Epoch'],
                   value_vars=metrics,
                   var_name='Metric',
                   value_name='Score')

# Grouped bar chart
fig, ax = plt.subplots(figsize=(20, 14))
sns.barplot(x='Metric', y='Score', hue='Epoch',
            data=melted_df, palette=epoch_palette,
            ax=ax, edgecolor='none')

ax.set_title('Comparison of Epochs across Metrics', fontsize=44, pad=30)
ax.set_xlabel('Evaluation Metric', fontsize=40, labelpad=25)
ax.set_ylabel('Score', fontsize=40, labelpad=25)
ax.set_ylim(0.75, 1.01)  # Adjusted for your data range

ax.legend(title='Epoch', title_fontsize=24, fontsize=22,
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout(pad=3.0)
save_figure(fig, "epoch_comparison_grouped_bar")

Saved: Experiment_Output_Figures/Epoch/epoch_comparison_grouped_bar.png and Experiment_Output_Figures/Epoch/epoch_comparison_grouped_bar.pdf


In [38]:
# Radar Chart
categories = metrics
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))
for i, epoch in enumerate(df['Epoch']):
    values = df.loc[i, metrics].values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=4, label=str(epoch),
            color=epoch_palette[epoch])
    ax.fill(angles, values, alpha=0.1, color=epoch_palette[epoch])

ax.set_ylim(0.75, 1.01)  # Adjusted for your data range
plt.xticks(angles[:-1], categories, fontsize=36)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=22)
plt.title('Epoch Comparison (Radar Chart)',
          fontsize=44, pad=40, y=1.08)

plt.tight_layout(pad=3.0)
save_figure(fig, "epoch_radar_chart")

Saved: Experiment_Output_Figures/Epoch/epoch_radar_chart.png and Experiment_Output_Figures/Epoch/epoch_radar_chart.pdf


In [39]:
# Heatmap
heatmap_df = df[['Epoch'] + metrics].set_index('Epoch')
fig, ax = plt.subplots(figsize=(18, 12))

heatmap = sns.heatmap(heatmap_df, annot=True, fmt=".4f", cmap="YlGnBu",
                     linewidths=0.5, linecolor='white', annot_kws={'size': 26},
                     vmin=0.75, vmax=1.0)  # Adjusted for your data range

cbar = heatmap.collections[0].colorbar
cbar.set_label('Score', size=34)
cbar.ax.tick_params(labelsize=28)

ax.set_title('Epoch Performance Heatmap', fontsize=40, pad=30)
ax.set_xlabel('Evaluation Metrics', fontsize=36, labelpad=25)
ax.set_ylabel('Epoch', fontsize=36, labelpad=25)

plt.tight_layout(pad=3.0)
save_figure(fig, "epoch_heatmap")

Saved: Experiment_Output_Figures/Epoch/epoch_heatmap.png and Experiment_Output_Figures/Epoch/epoch_heatmap.pdf


In [40]:
generate_violin_plot()

Saved: Experiment_Output_Figures/Epoch/epoch_violin_plot.png and Experiment_Output_Figures/Epoch/epoch_violin_plot.pdf


In [41]:
# generate_timing_comparison()

In [42]:
print("All epoch visualizations have been generated!")
print(f"Files are saved in the 'Epoch' folder")

All epoch visualizations have been generated!
Files are saved in the 'Epoch' folder
